# Model 5: Intermediate inputs and input-output linkages

In [38]:
import numpy as np
from scipy.optimize import root

## Functions

### Production

In [39]:
def output(phi, L_d, delta_l, delta_x, X_d):
    return phi * ((L_d / delta_l) ** delta_l) * ((X_d / delta_x) ** delta_x)

def output_value(M, b):
    return np.linalg.solve(np.eye(M.shape[0]) - M, b)

def unit_costs(phi, w, delta_l, delta_x, intermediate_price_index):
    return (1 / phi) * (w ** delta_l) * (intermediate_price_index ** delta_x)

def labour_demands(delta_l, output_value, w):
    return delta_l * output_value / w

### Prices

In [40]:
def price_factory_gate(w, A):
    return w / A

def price_ij(t, p):
    return p[:, np.newaxis] * (1 + t)

### Households

In [41]:
def income(w, L_bar):
    return w * L_bar

def armington_shares(p_ij, beta, eta_H):
    return (beta * p_ij ** (1 - eta_H)) / np.sum(beta * p_ij ** (1 - eta_H), axis=0)

def household_price_index(p_ij, beta, eta_H):
    return (np.sum(beta * p_ij ** (1 - eta_H), axis=0)) ** (1 / (1 - eta_H))

def household_bilateral_demand(s_H, I, p_ij):
    return s_H * I / p_ij

def household_real_consumption(I, P_H):
    return I / P_H

### Intermediate goods

In [42]:
def intermediate_shares(p_ij, gamma, eta_X):
    return (gamma * p_ij ** (1 - eta_X)) / np.sum(gamma * p_ij ** (1 - eta_X), axis=0)

def intermediate_price_index(p_ij, gamma, eta_X):
    return (np.sum(gamma * p_ij ** (1 - eta_X), axis=0)) ** (1 / (1 - eta_X))

def quantity_arriving_at_firm(s_X, delta_x, V, p_ij):
    return s_X * delta_x * V / p_ij

def intermediate_composite(gamma, eta_X, X_ij):
    return np.sum((gamma ** (1/eta_X)) * (X_ij ** ((eta_X - 1)/eta_X)), axis=0) ** (eta_X/(eta_X - 1))

### Final demands

In [43]:
def final_demand(armington_shares, income):
    return np.sum(armington_shares * income, axis=1)

def input_output(intermediate_shares, delta_x):
    return intermediate_shares * delta_x

### Market clearing / residuals

In [44]:
def goods_market_clearing(Y, t, Z, X_d):
    return Y - np.sum((1 + t) * (Z + X_d), axis=1)

def value_market_clearing(p, w, t, beta, gamma, eta_H, eta_X, phi, delta_l, delta_x, L_bar):
    p_ij = price_ij(t, p)
    P_H = household_price_index(p_ij, beta, eta_H)
    s_H = armington_shares(p_ij, beta, eta_H)
    P_X = intermediate_price_index(p_ij, gamma, eta_X)
    s_X = intermediate_shares(p_ij, gamma, eta_X)
    I = income(w, L_bar)

    b = final_demand(s_H, I)
    M = input_output(s_X, delta_x)

    V = output_value(M, b)

    # Calculate unit costs and labour demands
    c = unit_costs(phi, w, delta_l, delta_x, P_X)
    L_d = labour_demands(delta_l, V, w)

    R1 = p[0] - c[0]
    R2 = p[1] - c[1]
    R3 = L_d[1] - L_bar[1]

    return np.array([R1, R2, R3])

## Modelling values

In [45]:
L_bar = np.array([100, 100])
phi = np.array([1, 1])
delta_l = np.array([0.6, 0.6])
delta_x = np.array([0.4, 0.4])
beta = np.array([[0.6, 0.4], [0.4, 0.6]])
gamma = np.array([[0.6, 0.4], [0.4, 0.6]])
t = np.array([[0, 0.2], [0.2, 0]])
eta_H = 4
eta_X = 4

## Benchmark

In [46]:
def residuals(x):
    w = np.array([1.0, np.exp(x[2])])
    p = np.exp(x[:2])

    value_clearing = value_market_clearing(p, w, t, beta, gamma, eta_H, eta_X, phi, delta_l, delta_x, L_bar)

    return value_clearing

In [47]:
res = root(residuals, [0, 0, 0])

p1, p2, w2 = np.exp(res.x)

print("Solution found:", res.success)
print("Solution for w2:", w2)

Solution found: True
Solution for w2: 1.0000000000000002


In [48]:
w_sol = np.array([1.0, w2])
p_sol = np.array([p1, p2])

p_ij_sol = price_ij(t, p_sol)
P_H_sol = household_price_index(p_ij_sol, beta, eta_H)
s_H_sol = armington_shares(p_ij_sol, beta, eta_H)
P_X_sol = intermediate_price_index(p_ij_sol, gamma, eta_X)
s_X_sol = intermediate_shares(p_ij_sol, gamma, eta_X)
I_sol = income(w_sol, L_bar)

b_sol = final_demand(s_H_sol, I_sol)
M_sol = input_output(s_X_sol, delta_x)

V_sol = output_value(M_sol, b_sol)

L_d_sol = labour_demands(delta_l, V_sol, w_sol)

Q_sol = household_real_consumption(I_sol, P_H_sol)

print("Prices:", p_sol)
print("Wages:", w_sol)
print("Household Price Index:", P_H_sol)
print("Intermediate Price Index:", P_X_sol)
print("Labour Demands:", L_d_sol)
print("Output Values:", V_sol)
print("Real consumption:", Q_sol)
print("Household expenditure shares:", s_H_sol)

Prices: [1.04186281 1.04186281]
Wages: [1. 1.]
Household Price Index: [1.10796577 1.10796577]
Intermediate Price Index: [1.10796577 1.10796577]
Labour Demands: [100. 100.]
Output Values: [166.66666667 166.66666667]
Real consumption: [90.25549579 90.25549579]
Household expenditure shares: [[0.72160356 0.27839644]
 [0.27839644 0.72160356]]


### Verification of zero profit (price residuals = 0), labour-market clearing (differences between L_d and L_bar = 0) ...

In [49]:
# "Evaluate the residual vector R(x*) at the reported solution and verify that L_1^d - \bar{L}_1 ≈ 0."
print("Price and labour (L_2) residuals at solution:", residuals(res.x), "– Within 1.5e-14, so close to zero")

print("Difference between L_1^d and L_bar_1:", L_d_sol[0] - L_bar[0], "– Within 2.9e-14, so close to zero")

Price and labour (L_2) residuals at solution: [2.22044605e-16 0.00000000e+00 1.42108547e-14] – Within 1.5e-14, so close to zero
Difference between L_1^d and L_bar_1: 2.842170943040401e-14 – Within 2.9e-14, so close to zero


### ... and physical goods-market clearing (verification of equations (75))

In [50]:
X_ij_sol = quantity_arriving_at_firm(s_X_sol, delta_x, V_sol, p_ij_sol)
X_j_sol = intermediate_composite(gamma, eta_X, X_ij_sol)
Y_sol = output(phi, L_d_sol, delta_l, delta_x, X_j_sol)

Z_sol = household_bilateral_demand(s_H_sol, I_sol, p_ij_sol)

residual_goods_market = goods_market_clearing(Y_sol, t, Z_sol, X_ij_sol)

print("Total output:", Y_sol)
print("Goods Market clearing:", residual_goods_market, "– Within 2.9e-14, so close to zero")

Total output: [159.96987745 159.96987745]
Goods Market clearing: [2.84217094e-14 0.00000000e+00] – Within 2.9e-14, so close to zero


## Part 5.3: Verification of equation (82)

In [51]:
np.allclose(V_sol, np.linalg.solve(np.eye(M_sol.shape[0]) - M_sol, b_sol))

True

# TODO: Tasks 5 and 6

## Part VI: Comparison of models III–IV

In [52]:
L_bar = np.array([100, 100])
phi = np.array([1, 1])
delta_l = np.array([0.6, 0.6])
delta_x = np.array([0.4, 0.4])
beta = np.array([[0.6, 0.4], [0.4, 0.6]])
gamma = np.array([[0.6, 0.4], [0.4, 0.6]])
t = np.array([[0, 0.1], [0.1, 0]])
eta_H = 4
eta_X = 4

In [53]:
def residuals(x):
    w = np.array([1.0, np.exp(x[2])])
    p = np.exp(x[:2])

    value_clearing = value_market_clearing(p, w, t, beta, gamma, eta_H, eta_X, phi, delta_l, delta_x, L_bar)

    return value_clearing

In [54]:
res = root(residuals, [0, 0, 0])

p1, p2, w2 = np.exp(res.x)

print("Solution found:", res.success)
print("Solution for w2:", w2)

Solution found: True
Solution for w2: 1.0000000000000002


In [55]:
w_sol = np.array([1.0, w2])
p_sol = np.array([p1, p2])

p_ij_sol = price_ij(t, p_sol)
P_H_sol = household_price_index(p_ij_sol, beta, eta_H)
s_H_sol = armington_shares(p_ij_sol, beta, eta_H)
P_X_sol = intermediate_price_index(p_ij_sol, gamma, eta_X)
s_X_sol = intermediate_shares(p_ij_sol, gamma, eta_X)
I_sol = income(w_sol, L_bar)

b_sol = final_demand(s_H_sol, I_sol)
M_sol = input_output(s_X_sol, delta_x)

V_sol = output_value(M_sol, b_sol)

L_d_sol = labour_demands(delta_l, V_sol, w_sol)

Q_sol = household_real_consumption(I_sol, P_H_sol)

print("Prices:", p_sol)
print("Wages:", w_sol)
print("Household Price Index:", P_H_sol)
print("Intermediate Price Index:", P_X_sol)
print("Labour Demands:", L_d_sol)
print("Output Values:", V_sol)
print("Real consumption:", Q_sol)
print("Household expenditure shares:", s_H_sol)

Prices: [1.02355681 1.02355681]
Wages: [1. 1.]
Household Price Index: [1.05993658 1.05993658]
Intermediate Price Index: [1.05993658 1.05993658]
Labour Demands: [100. 100.]
Output Values: [166.66666667 166.66666667]
Real consumption: [94.34526769 94.34526769]
Household expenditure shares: [[0.66627732 0.33372268]
 [0.33372268 0.66627732]]


### Verification of zero profit (price residuals = 0), labour-market clearing (differences between L_d and L_bar = 0) ...

In [58]:
# "Evaluate the residual vector R(x*) at the reported solution and verify that L_1^d - \bar{L}_1 ≈ 0."
print("Price and labour (L_2) residuals at solution:", residuals(res.x), "– Within 2.3e-16, so close to zero")

print("Difference between L_1^d and L_bar_1:", L_d_sol[0] - L_bar[0], "– Within 1.5e-14, so close to zero")

Price and labour (L_2) residuals at solution: [ 0.00000000e+00 -2.22044605e-16  0.00000000e+00] – Within 2.3e-16, so close to zero
Difference between L_1^d and L_bar_1: 1.4210854715202004e-14 – Within 1.5e-14, so close to zero


### ... and physical goods-market clearing (verification of equations (75))

In [61]:
X_ij_sol = quantity_arriving_at_firm(s_X_sol, delta_x, V_sol, p_ij_sol)
X_j_sol = intermediate_composite(gamma, eta_X, X_ij_sol)
Y_sol = output(phi, L_d_sol, delta_l, delta_x, X_j_sol)

Z_sol = household_bilateral_demand(s_H_sol, I_sol, p_ij_sol)

residual_goods_market = goods_market_clearing(Y_sol, t, Z_sol, X_ij_sol)

print("Total output:", Y_sol)
print("Goods Market clearing:", residual_goods_market, "– Exactly zero.")

Total output: [162.83089052 162.83089052]
Goods Market clearing: [0. 0.] – Exactly zero.


### Verification of equation (82)

In [59]:
np.allclose(V_sol, np.linalg.solve(np.eye(M_sol.shape[0]) - M_sol, b_sol))

True